# 특성 공학과 규제
## 다중 회귀
이전 문서인 `03-2.ipynb` 에서는 하나의 특성을 이용하여 선형 회귀 모델을 훈련했다.\
이와는 다르게, 여러 개의 특성을 사용한 선형 회귀를 **다중 회귀(multiple regression)** 이라 한다.
> 하나의 특성에 대해 항이 여러 개인 다항 회귀와는 다르다.


이전처럼 하나의 특성을 사용했을 때, 회귀 모델이 학습하는 것은 하나의 직선이다.\
하지만, 2개의 특성을 사용하게 되면 3차원 공간의 평면을 학습한다.


특성이 2개면 3차원 공간을 학습하고, 그 공간에서 평면을 학습하여 회귀를 예상한다.
특성이 3개인 경우에는 4차원 공간을 예상할 수 있지만, 그 이상을 이해할 수는 없다.

특성이 많은 고차원일 경우에는 선형 회귀가 매우 복잡한 모델을 표현할 수 있게 된다.

이 예제에서는 농어의 길이, 무게뿐만 아니라 농어의 높이와 두께까지 모두 포함하겠다.
이전 절에서처럼 3개의 특성을 모두 제곱하여 추가한다.\
그 특성에서도 각 특성끼리 서로 곱하여 새 특성을 만들겠다.
>즉, `농어 길이`$\times$`농어 높이`를 새로운 특성으로 만드는 것이다.

### 특성 공학의 정의
위와 같이 기존의 속성을 사용하여 새로운 특성을 만들어내는 것을 **특성 공학(feature engineering)** 이라고 한다.

직접 특성을 변형하여 특성 공학을 할 수 있지만, 사용하는 라이브러리에서 특성 공학을 위한 도구를 제공하므로 그 도구를 이용한다.\
추가로, 많은 특성들을 사용하므로 그 배열 또한 고차원으로 바뀔 것이다.
>이전에도 확인하였듯이, 배열의 차원이 많아질수록 배열은 점점 복잡해진다.

이를 쉽게 하기 위헤 [pandas](https://pandas.pydata.org/) 프레임워크도 사용할 것이다.\
데이터를 다시 불러오는 것부터 시작하자.

본격적으로 모델 학습에 필요한 데이터를 준비해보자.
pandas에서 제공하는 **데이터프레임(dataframe)** 은 pandas의 핵심 데이터 구조이다.
해당 데이터프레임을 이용해 데이터를 관리하는 것이다.

넘파이의 배열과 비슷하게, 2차원 배열을 쉽게 관리할 수 있지만, 더 많은 기능을 제공한다.

이전에는 pandas 데이터프레임을 직접 sklearn 데이터에 입력할 수 없고, numpy 배열로 변경하여야 했지만,\
sklearn 1.2 업데이트부터는 pandas 데이터프레임을 입력 데이터로 사용할 수 있게 되었다.

pandas를 이용해 데이터를 내려받아 데이터프레임에 저장하겠다.\
그리고, 타깃 데이터는 따로 numpy 배열로 만들어 선형 회귀 모델을 훈련하겠다.

pandas dataframe를 만들기 위해 자주 사용하는 데이터 저장 형식은 csv 파일이다.\
데이터를 가져오자.

In [ ]:
import pandas as pd # 약어로 불러온다.
perch_full = pd.read_csv("https://raw.githubusercontent.com/rickiepark/hg-mldl/master/perch_full.csv") # https://raw.githubusercontent.com/rickiepark/hg-mldl/master/perch_full.csv
perch_full.head()

> pandas를 불러올 때에는 `pd`라는 별칭을 사용한다.

pandas의 `read_csv()` 메소드를 이용해 데이터셋을 가져올 수 있다.
> url 입력도 지원한다.
`head()` 메소드를 이용해 첫 5개의 값 부분을 확인할 수 있다.

데이터를 훈련할 때에는 위에서 생성된 csv인 `perch_full`을 사용하지만, 사이킷런 내부적으로 numpy 배열을 사용하므로 속도의 차이는 변환 속도밖에는 없다.\
이제 나머지 타깃 데이터를 가져오자.

In [ ]:
import numpy as np
perch_weight = np.array([5.9, 32.0, 40.0, 51.5, 70.0, 100.0, 78.0, 80.0, 85.0, 85.0, 110.0,
       115.0, 125.0, 130.0, 120.0, 120.0, 130.0, 135.0, 110.0, 130.0,
       150.0, 145.0, 150.0, 170.0, 225.0, 145.0, 188.0, 180.0, 197.0,
       218.0, 300.0, 260.0, 265.0, 250.0, 250.0, 300.0, 320.0, 514.0,
       556.0, 840.0, 685.0, 700.0, 700.0, 690.0, 900.0, 650.0, 820.0,
       850.0, 900.0, 1015.0, 820.0, 1100.0, 1000.0, 1100.0, 1000.0,
       1000.0])

다음, 입력 데이터와 타깃 데이터를 테스트 세트와 훈련 세트로 나눈다.

In [ ]:
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(perch_full, perch_weight, random_state=42)

생성된 이 데이터를 활용해 새 특성을 만들겠다.(데이터 공학)

## 사이킷런의 변환기
사이킷런에서 특성 공학이나 데이터 전처리에 사용되는 클래스들을 변환기(transformer)라고 부른다.\
모든 모델 관련 클래스에 `fit()`, `predict()`, `score()`이 있는 것처럼,\
변환기 클래스에는 `fit()`, `transform()` 메소드를 제공한다.
> 선형 회귀(`LinearRegression`)는 추정기(estimator)에 속한다.

사용할 변환기는 `PolynomialFeatures` 클래스이다.\
sklearn의 `preprocessing` 모듈(전처리 모듈)에 포함되어 있다.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

예시를 들어, 2개의 특성 `2`와 `3`으로 이루어진 샘플 하나를 사용하겠다.\
이 클래스의 인스턴스를 생성한 후, `fit()`과 `transform()`을 차례대로 호출한다.

In [ ]:
poly = PolynomialFeatures(include_bias=True)
poly.fit([[2, 3]]) # 입력값은 2차원 배열
print(poly.transform([[2, 3]]))

사이킷런은 일관된 클래스를 상속받아 `PolynomialFeatures()` 클래스를 사용할 때에도 `fit()` 메소드를 이용해 데이터를 입력받는 구조이다.
> `fit()`으로 데이터를 입력하고 다시 출력하지 않고 두 작업을 하나로 합친 메소드인 `fit_transform()` 메소드도 존재한다.

`PolynomialFeatures()` 클래스는 변환기 이므로 훈련 데이터만 입력받고 목표 데이터(타깃 데이터)는 입력받지 않았다.\
즉, 이 변환기에서는 2개의 특성을 가진 샘플 `[2, 3]`이 6개의 특성을 가진 샘플인 `[1, 2, 3, 4, 6, 9]`로 변환되었다.

**기존의 샘플 데이터를 곱하거나 제곱하여 새로운 샘플 데이터를 만들어낸 것이다.**

`PolynomialFeatures()` 클래스는 기본적으로 각 특성을 제곱한 항을 추가하고 특성끼리 서로 곱한 항을 추가한다.\
2와 3을 각기 제곱한 4와 9가 추가되었고, 2와 3을 곱한 6 또한 추가되었다.
> 예외적으로, 1이 추가되었는데 이는 절편을 학습하기 위한 상수항이다.
> 선형 회귀 모델의 선형 방정식의 절편을 학습하기 위해 상수항 값인 1을 추가한 것이다.\
> 다만, 선형 회귀 모델인 `LinearRegression()` 클래스는 방정식의 상수항 값을 자동으로 추가하기 때문에 일반적으로는 필요하지 않다.
> (코드에는 `include_bias=False` 매개변수로 명시적으로 없앨 수 있다.)


In [ ]:
poly = PolynomialFeatures(include_bias=False) # bias 포함 제외
poly.fit([[2, 3]])
print(poly.transform([[2, 3]]))

1이 제외된 값이 출력된다.\
이제 모델의 학습 데이터에 직접 적용해보자.

In [ ]:
poly = PolynomialFeatures(include_bias=False) # include_bias 매개변수는 포함하지 않아도 된다.
poly.fit(train_input)
train_poly = poly.transform(train_input)
print(train_poly.shape) # 배열의 형태 확인

배열을 변환했다.\
변환된 배열의 값들이 어떻게 만들어졌는지 확인하려면 `get_feature_names_out()` 메소드를 이용한다.

In [ ]:
poly.get_feature_names_out()

배열의 값들이 제곱인지 두 값을 곱하여 만들어졌는지 확인할 수 있다.\
테스트 세트를 변환해보자.

In [ ]:
test_poly = poly.transform(test_input)

변환된 두 세트들을 활용해 모델을 학습하겠다.

## 다중 회귀 모델 훈련
훈련 세트로 다중 회귀 모델을 훈련한다.
다중 회귀 모델을 훈련하는 것은 이전의 선형 회귀 모델을 훈련하는 것과 같다.\
다만, 여러 개의 특성을 사용하여 선형 회귀를 수행한다는 점이 다르다.\

In [ ]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(train_poly, train_target)
print(lr.score(train_poly, train_target))

점수가 상당히 높게 책정되었다.\
이를 통해 선형 회귀 모델은 특성이 많아져 복잡해질수록 정확도가 높아짐을 알 수 있다.\
테스트 세트의 점수는:

In [ ]:
print(lr.score(test_poly, test_target))

이전보다는 나은 점수를 가지고 있다.

데이터 전처리 과정의 특성 공학 과정에서의 생성되는 값들의 수를 늘일수록 선형 회귀 모델의 정확도는 높아진다.
테스트 세트에 대한 점수는 개선되지 않았지만, 이전의 과소적합 문제는 해결되었다.

특성을 더 많이 추가하여 곱하기, 제곱된 특성 뿐만아니라 3제곱 이상의 특성까지 포함하게 하여 모델을 훈련시켜 보자.
`PolynomialFeatures` 클래스의 `degree` 매개변수를 수정하여 이를 구현할 수 있다.

In [ ]:
poly = PolynomialFeatures(include_bias=False, degree=5) # 5제곱까지
poly.fit(train_input)

train_poly_high = poly.transform(train_input)
test_poly_high = poly.transform(test_input)

print(train_poly_high.shape) # 배열의 형태 확인
print(test_poly_high.shape)

만들어진 특성의 개수가 55개이다.
> 배열의 열 의 개수가 특성의 개수이다.

이를 이용해 모델을 다시 훈련해본다면:

In [ ]:
lr.fit(train_poly_high, train_target)
print(lr.score(train_poly_high, train_target))

훈련한 값으로 $R^2$ 점수를 확인했을 때에는 매우 높은 점수가 나온다.
테스트 세트의 경우는:

In [ ]:
print(lr.score(test_poly_high, test_target))

테스트 세트의 점수는 음수가 나온다.

특성의 개수를 크게 늘이면 선형 모델은 아주 높은 성능을 보인다.\
훈련 세트에 대해 거의 완벽하게 학습이 가능하다.\
다만, 이런 모델은 훈련 세트에 과다하게 의존하여 훈련 세트의 특성만 따라가는 경우가 많다.
> 이로 인해 **과대적합(overfitting)** 이 발생하게 되는 것이다.

이번에 만든 모델처럼 이러한 모델은 훈련 세트에 과다하게 과대적합하여서 테스트 세트에서는 매우 낮은 점수를 보인다.
이 문제를 해결하려면 다시 모델의 특성을 줄여야 하지만, 이번 예시를 이용해 과대적합을 줄이는 방법을 알아볼 것이다.

# 규제
**규제(regularization)** 은 머신러닝 모델이 훈련 세트를 과도하게 학습하여 과대적합이 일어나는 것을 막는 것을 말한다.

선형 회귀의 경우 특성 전체를 그래프로 표현하였을 때 훈련 세트의 특성에 맞는 그래프의 기울기를 낮추어 샘플의 전체적인 특성을 포함하게 한다.
> 쉽게 설명해서 특성에 곱해지는 계수(기울기)를 작게 만드는 것이다.

이해를 돕기 위해 AI를 사용하여 예시 그래프를 생성해보았다.\
첫 번째 그래프는 훈련 세트를 과도하게 학습한 그래프이고, 두 번째 그래프는 첫 번째 모델에 규제를 적용하여 보다 보편적인 페턴을 학습하고 있는 선형 회귀 모델이다.

In [ ]:
import matplotlib.pyplot as plt

# 데이터 범위 설정
x = np.linspace(0, 50, 300)

# 과대적합 모델 (복잡한 곡선)
y_overfitting = 50 + 0.8*x + 0.05*x**2 - 0.001*x**3 + np.random.normal(0, 10, 300)

# 규제가 적용된 모델 (매끄러운 직선)
y_regularized = 50 + 0.8*x + np.random.normal(0, 15, 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 첫 번째 그래프: 과대적합
axes[0].scatter(x, y_overfitting, alpha=0.5, s=20, color='blue', label='훈련 데이터')
z_overfit = np.polyfit(x, y_overfitting, 5)
p_overfit = np.poly1d(z_overfit)
axes[0].plot(x, p_overfit(x), color='red', linewidth=2, label='과대적합 모델')
axes[0].set_xlabel('특성 (x)', fontsize=12)
axes[0].set_ylabel('예측값 (y)', fontsize=12)
axes[0].set_title('규제 없음 (과대적합)', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 두 번째 그래프: 규제 적용
axes[1].scatter(x, y_regularized, alpha=0.5, s=20, color='blue', label='훈련 데이터')
z_reg = np.polyfit(x, y_regularized, 1)
p_reg = np.poly1d(z_reg)
axes[1].plot(x, p_reg(x), color='green', linewidth=2, label='규제 적용 모델')
axes[1].set_xlabel('특성 (x)', fontsize=12)
axes[1].set_ylabel('예측값 (y)', fontsize=12)
axes[1].set_title('규제 적용 (일반화)', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

앞서 55개의 특성을 이용해 훈련한 모델의 기울기 계수를 규제하여 훈련 세트의 점수를 희생하고 테스트 세트의 점수를 높여 보겠다.\

그 전, 특성의 스케일에 대해 확인해보자.
이전 문서에도 설명하였듯이, 특성의 스케일이 정규화되지 않으면 곱해지는 계수 값도 변형된다.\
일반적으로 선형 회귀 모델에 규제를 적용할 때 계수 값의 크기가 서로 많이 다르면 공정하게 제어되지 않을 것이다.
> 이전 문서에서 설명할 때 x축과 y 축의 스케일 차이가 커서 knn 모델이 제대로 작동하지 않았다.
> 해당 문서에서는 평균과 표준편차를 직접 구해 표준점수를 이용해 스케일 차이를 안정화했다.

이 방법이 크게 어렵지는 않지만, 이미 사이킷런이 제공하고 있는 천처리 기능이 존재한다.\
`StanadardScaler` 변환기 클래스는 샘플의 스케일을 맞추어 준다.

In [ ]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
ss.fit(train_poly_high)
train_poly_high_scaled = ss.transform(train_poly_high)
test_poly_high_scaled = ss.transform(test_poly_high)

`StandardScaler` 클래스를 불러와 초기화한 후 `ss`에 인스턴스를 생성하고 이를 `train_poly_high`로 객체를 훈련한다.
이후 스케일된 값들을 각각 `train_poly_high_scaled`와 `test_poly_high_scaled`에 저장했다.

이제 표준점수로 변환된 샘플들이 모두 준비되었다.
> 이전 문서에서 직접 평균과 표준 편차를 구하고 표준점수를 직접 계산하였지만, 미리 제공된 변환기를 이용해 쉽게 변환이 가능하다.

> `StandardScaler`의 훈련 세트에 학습한 평균과 표준편차는 `StandardScaler` 클래스(또는 그 인스턴스)의 `mean_`과 `scale_` 속성에 저장된다.

## 릿지(ridge)와 라쏘(lasso)
선형 회귀 모델에 규제를 추가한 모델을 릿지(ridge)와 라쏘(lasso)라고 부른다.
> 선형 회귀류 모델에만 포함된다.

두 모델은 모델에 규제를 가하는 방식이 다르다.\
릿지는 계수를 제곱한 값 중심으로 규제를 적용하고, 라쏘는 계수의 절댓값을 기준으로 규제를 적용한다.
> 일반적으로 릿지를 조금 더 선호한다.
> 두 알고리즘 모두 계수의 크기를 줄이지만, 계수의 절댓값을 중심으로 하는 라쏘의 경우 계수를 완전히 0으로 만들어버릴 가능성이 존재하기 때문이다.

각각의 회귀 방식과 규제 방법에 대해 자세히 알아보자.

> 릿지와 라쏘 모델 모두 `sklearn.linear_model` 패키지 내에 존재한다.


## 릿지(ridge) 회귀
`sklearn.linear_model` 패키지에 `Ridge` 라는 클래스 이름으로 불러온다.\
일반적인 선형 회귀 모델인 `LinearRegression`에 릿지 규제를 적용한 모델이다.

스케일된 샘플을 이용해 모델을 학습하고 평가를 매기면:

In [ ]:
from sklearn.linear_model import Ridge
ridge = Ridge()
ridge.fit(train_poly_high_scaled, train_target)
print(ridge.score(train_poly_high_scaled, train_target))

테스트 세트로 평가하면:

In [ ]:
print(ridge.score(test_poly_high_scaled, test_target))

과대적합을 줄여 테스트 세트의 점수가 정상화되었다.\

릿리와 라쏘 모델을 사용할 때 규제량을 `alpha` 매개변수를 활용해 임의로 조절할 수 있다.
> 언급되는 매개변수는 대부분 **하이퍼파라미터(hyperparameter)** 이다.
> 사용자가 임의로 조절할 수 있는 매개변수이다.(모델 파라미터와는 다르다.)

모델에 대한 적절한 `alpha` 값을 찾는 방법은 각 `alpha` 값에 대한 테스트 세트와 훈련 세트의 $R^2$ 값의 그래프를 그려 차이가 가장 작은 곳이 가장 적절한 `alpha` 값이라 볼 수 있다.

In [ ]:
import matplotlib.pyplot as plt
train_score_ridge = []
test_score_ridge = []

`alpha` 값을 `0.001`부터 시작하여 10배씩 늘려가며 릿지 모델을 훈련한 다음 훈련 세트와 테스트 세트의 점수를 그래프로 그린다.

In [ ]:
alpha_list = [0.001, 0.01, 0.1, 1, 10, 100]
for alpha in alpha_list:
    ridge_alpha = Ridge(alpha=alpha)
    # 릿지 모델 훈련
    ridge_alpha.fit(train_poly_high_scaled, train_target)
    # 훈련 세트의 점수와 테스트 세트의 점수 저장
    train_score_ridge.append(ridge_alpha.score(train_poly_high_scaled, train_target))
    test_score_ridge.append(ridge_alpha.score(test_poly_high_scaled, test_target))

이제 그래프를 그려보자.\
다만, `alpha`값을 `0.001` 부터 늘였기 때문에 처음 3개의 인덱스는 1보다 작아 그래프의 왼쪽이 촘촘해진다.\
6개의 값은 각각 10의 배수 차이이므로 10의 제곱수로 표현이 가능하다. 따라서, 로그 스케일로 표현할 수 있다.\
`matplotlib`은 `xscale()`와 `yscale()` 함수로 간단히 축의 눈금을 로그 스케일로 변환할 수 있다.

In [ ]:
plt.plot(alpha_list, train_score_ridge)
plt.plot(alpha_list, test_score_ridge)
plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('score R^2')
plt.show()

파란색은 훈련 세트 그래프, 주황색은 훈련 세트의 그래프이다.
이 그래프의 왼쪽은 훈련 세트와 테스트 세트의 점수가 매우 크다.(과대적합)\
반대로 오른쪽은 훈련 세트와 테스트 세트 점수 모두 낮아지는 과소적합을 보인다.\

적절한 `alpha` 값은 두 그래프의 $R^2$ 값의 차이가 가장 작은 $10^-1 = 0.1$ 이다.\
`alpha` 값을 `0.1`로 하여 모델을 훈련해보면:

In [ ]:
best_ridge = Ridge(alpha=0.1)
best_ridge.fit(train_poly_high_scaled, train_target)
print(best_ridge.score(train_poly_high_scaled, train_target))
print(best_ridge.score(test_poly_high_scaled, test_target))

## 라쏘 회귀
라쏘 회귀 모델은 `linear_model`의 `Lasso` 클래스이다.\

In [ ]:
from sklearn.linear_model import Lasso
lasso = Lasso()
lasso.fit(train_poly_high_scaled, train_target)
print(lasso.score(train_poly_high_scaled, train_target))
print(lasso.score(test_poly_high_scaled, test_target))

라쏘 모델 또한 과대적합을 잘 억제한 모습을 보인다.\
테스트 세트의 점수도 좋다.

라쏘 모델 또한 릿지 모델과 같이 `alpha` 파라미터로 규제 적용 정도를 변경할 수 있다.\
마찬가지로 각 `alpha` 값에 대한 모델의 점수를 비교하며 가장 적절한 값을 찾겠다.
> 추가로, `Lasso` 클래스는 최적의 계수를 찾기 위해 반복적인 반복문 연산을 수행하는데, 지정한 반복 횟수가 부족할 때 `ConvergenceWarning` 예외가 발생한다.
> 이를 방지하기 위해 반복 횟수를 지정하는 `max_iter` 매개변수를 `10000` 으로 늘였다.(필요하다면 더 늘일 수 있지만, 이런 간단한 모델에서는 필요하지 않다.)(기본값은 1000)

In [ ]:
train_score_lasso = []
test_score_lasso = []

alpha_list = [0.001, 0.01, 0.1, 1, 10, 100] # 재선언
for alpha in alpha_list:
    # 라쏘 모델 생성
    lasso_alpha = Lasso(alpha=alpha, max_iter=10000)
    # 라쏘 모델 훈련
    lasso_alpha.fit(train_poly_high_scaled, train_target)
    # 라쏘 모델 점수 저장
    train_score_lasso.append(lasso_alpha.score(train_poly_high_scaled, train_target))
    test_score_lasso.append(lasso_alpha.score(test_poly_high_scaled, test_target))

In [ ]:
plt.plot(alpha_list, train_score_lasso)
plt.plot(alpha_list, test_score_lasso)
plt.xscale('log')
plt.xlabel("alpha")
plt.ylabel("R^2")
plt.show()

파란색 그래프는 훈련 세트 그래프이고, 아래의 주황색 그래프는 테스트 세트 그래프이다.\
이 그래프들도 왼쪽은 과대적합을 보여주고 있고, 오른쪽은 두 점수 모두 크게 감소하는 과소적합을 보여준다.

그래프에 따르면, 라쏘 모델에서의 최적의 `alpha` 값은 $10^1 = 10$ 이다.

In [ ]:
best_lasso = Lasso(alpha=10)
best_lasso.fit(train_poly_high_scaled, train_target)
print(best_lasso.score(train_poly_high_scaled, train_target))
print(best_lasso.score(test_poly_high_scaled, test_target))

모델이 적절히 훈련되었다.
특성을 더 많이 사용했지만, 릿지와 마찬가지로 라쏘 모델도 규제를 이용해 과대적합을 잘 억제하고 테스트 세트의 성능을 높였다.\
앞서 라쏘 모델은 계수 값을 완전히 0으로 만들 가능성이 있다고 언급하였다.\
라쏘 모델의 계수는 `coef_` 속성에 저장되어 있다. 해당 모델 중 계수가 0인 것만 확인해보면:

In [ ]:
print(np.sum(best_lasso.coef_ == 0))

> `np.sum()` 함수는 배열의 모든 값을 더한 값을 반환한다.
> 넘파이 배열에 비교 연산자를 사용했을 때에는 각 원소의 값이 불 값으로 변하는데, 넘파이는 내부적으로 불을 정수로 변환할 때 긍정을 1로, 부정을 0으로 표현하므로 마치 비교 연산자에 맞는 원소 개수를 세는 효과를 낸다.

55개의 특성을 모델에 주었지만, 모델은 계수가 0이 된 40개를 제외한 15개만 모델이 학습하였다.\
이런 특징 때문에 라쏘 모델은 유용한 특성을 골라내는 용도로도 사용된다.

# Recap
- 선형 회귀 알고리즘을 사용해 농어의 무게를 예측하는 모델 훈련
- 훈련 세트의 과소적합을 훈련하기 위해 더 다양한 특성을 사용
- 특성 공학으로 다항 특성을 사용해 훈련 세트에서의 정확도 증가 => 하지만 과대적합 발생
- 선형 회귀에 규제 적용 필요, 규제를 적용한 릿지(Ridge)와 라쏘(Lasso) 회귀 모델
- 두 모델의 최적의 규제 정도(`alpha`)값 구하기